In [1]:
import subprocess
import time

# Start MLflow server in background if not already running
mlflow_server = subprocess.Popen(
    ["uv", "run", "mlflow", "ui", "--port", "5000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(3)  # Give it time to start up
print("✅ MLflow server started at http://localhost:5000")

✅ MLflow server started at http://localhost:5000


In [31]:
import duckdb
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import shap
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (classification_report, roc_auc_score, 
                              confusion_matrix, PrecisionRecallDisplay,
                              precision_recall_curve, average_precision_score)
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

# Point MLflow at your local server
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("olist-churn-prediction")

print("✅ Setup complete")

✅ Setup complete


In [32]:
con = duckdb.connect('../data/olist.db')
print(f"Connected ✅")
print(con.execute("SHOW TABLES").df().to_string())

Connected ✅
                    name
0   category_translation
1              customers
2            geolocation
3            order_items
4          order_payment
5         order_payments
6          order_reviews
7                 orders
8                product
9               products
10               sellers


Number of orders that each unique customer has made.

In [33]:
query = """
WITH
  base_orders AS (
    SELECT
      customers.customer_unique_id,
      orders.order_id,
      orders.order_purchase_timestamp
    FROM
      customers
      LEFT JOIN orders ON customers.customer_id = orders.customer_id
    WHERE
      orders.order_status = 'delivered'
  ),
  customer_metrics AS (
    SELECT
      base_orders.customer_unique_id,
      COUNT(DISTINCT base_orders.order_id) AS total_orders,
      SUM(order_items.price + order_items.freight_value) AS total_spent,
      AVG(order_items.price + order_items.freight_value) AS avg_order_value,
      MIN(base_orders.order_purchase_timestamp) AS first_order_date,
      MAX(base_orders.order_purchase_timestamp) AS last_order_date,
      COUNT(DISTINCT order_items.product_id) AS unique_product_bought,
      COUNT(DISTINCT order_items.seller_id) AS unique_sellers_used
    FROM
      base_orders
      LEFT JOIN order_items ON base_orders.order_id = order_items.order_id
    GROUP BY
      base_orders.customer_unique_id
  ),
  review_metrics AS (
    SELECT
      base_orders.customer_unique_id,
      AVG(order_reviews.review_score) AS avg_review_score
    FROM
      base_orders
      LEFT JOIN order_reviews ON base_orders.order_id = order_reviews.order_id
    GROUP BY
      base_orders.customer_unique_id
  ),
  payment_metrics AS (
    SELECT
      base_orders.customer_unique_id,
      AVG(order_payments.payment_installments) AS avg_installments,
      COUNT(DISTINCT order_payments.payment_type) as payment_types_used
    FROM
      base_orders
      LEFT JOIN order_payments ON order_payments.order_id = base_orders.order_id
    GROUP BY
      base_orders.customer_unique_id
  ),
  final_metrics AS (
    SELECT
      customer_metrics.customer_unique_id,
      customer_metrics.total_orders,
      customer_metrics.total_spent,
      customer_metrics.avg_order_value,
      customer_metrics.first_order_date,
      customer_metrics.last_order_date,
      customer_metrics.unique_product_bought,
      customer_metrics.unique_sellers_used,
      review_metrics.avg_review_score,
      payment_metrics.avg_installments,
      payment_metrics.payment_types_used,
      DATEDIFF (
        'day',
        last_order_date,
        (
          SELECT
            MAX(orders.order_purchase_timestamp)
          FROM
            orders
        )
      ) AS recency_days,
      DATEDIFF (
        'day',
        customer_metrics.first_order_date,
        customer_metrics.last_order_date
      ) AS customer_lifespan_days
    FROM
      customer_metrics
      LEFT JOIN review_metrics ON review_metrics.customer_unique_id = customer_metrics.customer_unique_id
      LEFT JOIN payment_metrics ON payment_metrics.customer_unique_id = customer_metrics.customer_unique_id
  )
SELECT
  *,
  CASE
    WHEN recency_days > 90 THEN 1
    ELSE 0
  END AS churned
FROM
  final_metrics
"""

df = con.execute(query).df()

print(f"Dataset shape:      {df.shape}")
print(f"Churn rate:         {df['churned'].mean():.2%}")
print(f"Churned:            {df['churned'].sum():,}")
print(f"Not churned:        {(df['churned']==0).sum():,}")
print(f"\nMissing values:")
print(df.isnull().sum())
print(f"\nSample:")
df.head()

Dataset shape:      (93358, 14)
Churn rate:         89.98%
Churned:            83,999
Not churned:        9,359

Missing values:
customer_unique_id          0
total_orders                0
total_spent                 0
avg_order_value             0
first_order_date            0
last_order_date             0
unique_product_bought       0
unique_sellers_used         0
avg_review_score          603
avg_installments            1
payment_types_used          0
recency_days                0
customer_lifespan_days      0
churned                     0
dtype: int64

Sample:


,customer_unique_id,total_orders,total_spent,avg_order_value,first_order_date,last_order_date,unique_product_bought,unique_sellers_used,avg_review_score,avg_installments,payment_types_used,recency_days,customer_lifespan_days,churned
0,b2c72d1e9f6430603b8337d8f1394a99,1,41.11,41.11,2018-02-22 11:54:42,2018-02-22 11:54:42,1,1,3.0,4.0,1,237,0,1
1,7ac26eac431c6848694a2de6f0327524,1,135.41,135.41,2017-03-23 12:21:17,2017-03-23 12:21:17,1,1,5.0,1.0,1,573,0,1
2,98e71752819916789567071c52c4239e,1,186.19,186.19,2018-03-10 18:53:06,2018-03-10 18:53:06,1,1,1.0,2.0,1,221,0,1
3,a666bec8af192f2238a0b719a7063a00,1,177.01,177.01,2018-04-25 21:47:26,2018-04-25 21:47:26,1,1,5.0,4.0,1,175,0,1
4,f26b89209f5378f95d8e664073b76419,1,94.74,94.74,2018-08-19 19:43:01,2018-08-19 19:43:01,1,1,4.0,3.0,1,59,0,0


In [34]:
# In this cell we are doing some pre-processing. 

# 1. Droping the customer_unique_id, first_order_date, last_order_date
df = df.drop(columns=['customer_unique_id', 'first_order_date', 'last_order_date', 'recency_days'])

# 2. dropping the missing 
df = df[~df['avg_installments'].isnull()]
df = df[~df['avg_review_score'].isnull()]

# 3. Defining Feature cols
feature_cols = df.columns.drop('churned')
X = df[feature_cols]
y = df['churned']

# 4. Train / Test Split - stratified, 80/20, random_state = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Smote for training data only
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

In [35]:
# Training the Random Forest Algorithm

rf_params = {
    "n_estimators": 300,
    "max_depth": 8,
    "min_samples_leaf": 20,
    "class_weight": "balanced",
    "random_state": 42,
    "n_jobs": -1,
}

with mlflow.start_run(run_name="RandomForest_v1"):

    # 1. Train
    model = RandomForestClassifier(**rf_params)
    model.fit(X_train_bal, y_train_bal)

    # 2. Predict
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    # 3. Score
    roc_auc = roc_auc_score(y_test, y_proba)

    # 4. Log to MLflow
    mlflow.log_params(rf_params)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.sklearn.log_model(model, "random_forest_model")

    print(f"ROC-AUC: {roc_auc:.4f}")
    print(classification_report(y_test, y_pred))

2026/05/13 14:53:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/13 14:53:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/13 14:53:40 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


ROC-AUC: 0.5278
              precision    recall  f1-score   support

           0       0.11      0.47      0.18      1864
           1       0.91      0.56      0.69     16687

    accuracy                           0.55     18551
   macro avg       0.51      0.52      0.43     18551
weighted avg       0.82      0.55      0.64     18551

🏃 View run RandomForest_v1 at: http://localhost:5000/#/experiments/1/runs/96efefba26c84db48bc7fb7fe082935b
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [36]:
feature_importance = pd.DataFrame({
    'feature': X_train_bal.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance)

                  feature  importance
5        avg_review_score    0.580059
6        avg_installments    0.219690
7      payment_types_used    0.058738
1             total_spent    0.041498
2         avg_order_value    0.035318
8  customer_lifespan_days    0.032225
3   unique_product_bought    0.015760
0            total_orders    0.010735
4     unique_sellers_used    0.005978
